# Metrics

In [1046]:
import pandas as pd
import ast
from sklearn.metrics import multilabel_confusion_matrix
import numpy as np
from collections import defaultdict, Counter


In [1047]:
def compute_fpr(df):
    true_label_classes = sorted(df['true_label'].unique())
    
    fpr_dict = {}

    for label in true_label_classes:
        tp = ((df['true_label'] == label) & (df['predicted_label'] == label)).sum()
        fp = ((df['true_label'] != label) & (df['predicted_label'] == label)).sum()
        tn = ((df['true_label'] != label) & (df['predicted_label'] != label)).sum()

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fpr_dict[label] = fpr

    return fpr_dict


In [1048]:
def compute_full_match_per_group(df):
    group_acc = {}

    grouped = df.groupby('true_label')

    for label, group in grouped:
        matches = (group['true_label'] == group['predicted_label']).sum()
        total = len(group)
        acc = matches / total
        group_acc[label] = acc

    return group_acc

In [1049]:
def compute_partial_but_not_full_match(df):
    from collections import defaultdict

    group_scores = defaultdict(list)

    for _, row in df.iterrows():
        true_set = set(row['true_label'].split('-'))
        pred_set = set(row['predicted_label'].split('-'))

        if true_set == pred_set:
            continue

        intersection = true_set.intersection(pred_set)
        union = true_set.union(pred_set)

        if intersection:  # Only consider partial matches (non-empty intersection)
            iou = len(intersection) / len(union)
            group_scores[row['true_label']].append(iou)

    group_partial_not_full = {label: sum(scores)/len(scores) for label, scores in group_scores.items()}
    return group_partial_not_full


In [1050]:
def most_frequent_language_in_predictions(df):
    group_lang_counts = defaultdict(Counter)

    for _, row in df.iterrows():
        label_group = row['true_label']
        true_langs = label_group.split('-')
        predicted = row['predicted_label'].split('-')

        if sorted(true_langs) != sorted(predicted):
            for lang in true_langs:
                if lang in predicted:
                    group_lang_counts[label_group][lang] += 1

    group_top_lang = {
        group: lang_count.most_common(1)[0][0] if lang_count else None
        for group, lang_count in group_lang_counts.items()
    }

    return group_top_lang


In [1051]:
test_csv = pd.read_csv('test_combined_separated_cs_dataset.csv')

In [1052]:
true_labels = test_csv['languages']

## OpenLID

In [1053]:
openlid_results = pd.read_csv("../../dataset/code-switch/codeswitch_baselines_results/codeswitch/OpenLID_multi_cs.csv")

In [1054]:
openlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob
0,Yellam Avan seyal . . .,English-Tamil,"{'zho_Hant': 0.5538930892944336, 'zho_Hans': 0...",zho_Hant,0.553893
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.9959315061569214, 'hrv_Latn': 0...",eus_Latn,0.995932
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.9779373407363892, 'swh_Latn': 0...",eng_Latn,0.977937
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'yor_Latn': 0.19660769402980804, 'hau_Latn': ...",yor_Latn,0.196608
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'zho_Hans': 0.9475940465927124, 'zho_Hant': 0...",zho_Hans,0.947594
...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.8605847358703613, 'dan_Latn': 0...",eng_Latn,0.860585
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.30808666348457336, 'est_Latn': ...",eng_Latn,0.308087
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'zho_Hans': 0.9955909848213196, 'zho_Hant': 0...",zho_Hans,0.995591
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 1.0000077486038208, 'ajp_Arab': 1...",arz_Arab,1.000008


In [1055]:
language_to_iso_script = {
    'Arabizi': 'arb_Latn',
    'Basque': 'eus_Latn',
    'Chinese': 'zho_Hans', 
    'Chinese': 'zho_Hant', 
    'Egyptian': 'arz_Arab',
    'English': 'eng_Latn',
    'German': 'deu_Latn',
    'Hindi': 'hin_Deva',
    'Indonesian': 'ind_Latn',
    'MSA': 'arb_Arab',       
    'Malayalam': 'mal_Mlym',
    'Saudi': 'ars_Arab',
    'Spanish': 'spa_Latn',
    'Tamil': 'tam_Taml',
    'Turkish': 'tur_Latn'
}

In [1056]:
def map_to_iso_pair(lang_pair):
    langs = lang_pair.split('-')
    return [language_to_iso_script.get(lang, 'unknown') for lang in langs]

In [1057]:
openlid_results['iso_pair'] = openlid_results['languages'].apply(map_to_iso_pair)

In [1058]:
openlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair
0,Yellam Avan seyal . . .,English-Tamil,"{'zho_Hant': 0.5538930892944336, 'zho_Hans': 0...",zho_Hant,0.553893,"[eng_Latn, tam_Taml]"
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.9959315061569214, 'hrv_Latn': 0...",eus_Latn,0.995932,"[eus_Latn, spa_Latn]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.9779373407363892, 'swh_Latn': 0...",eng_Latn,0.977937,"[eng_Latn, hin_Deva]"
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'yor_Latn': 0.19660769402980804, 'hau_Latn': ...",yor_Latn,0.196608,"[eng_Latn, tam_Taml]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'zho_Hans': 0.9475940465927124, 'zho_Hant': 0...",zho_Hans,0.947594,"[zho_Hant, eng_Latn]"
...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.8605847358703613, 'dan_Latn': 0...",eng_Latn,0.860585,"[eng_Latn, hin_Deva]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.30808666348457336, 'est_Latn': ...",eng_Latn,0.308087,"[eng_Latn, tam_Taml]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'zho_Hans': 0.9955909848213196, 'zho_Hant': 0...",zho_Hans,0.995591,"[zho_Hant, eng_Latn]"
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 1.0000077486038208, 'ajp_Arab': 1...",arz_Arab,1.000008,"[arz_Arab, eng_Latn]"


In [1059]:
def get_top2_iso(pred_lang_str):
    try:
        pred_lang_dict = ast.literal_eval(pred_lang_str)
        top2 = sorted(pred_lang_dict.items(), key=lambda x: x[1], reverse=True)[:2]
        top2_langs = [lang for lang, _ in top2]
        return top2_langs
    except Exception as e:
        print(f"Error parsing: {pred_lang_str} — {e}")
        return []

In [1060]:
import ast

openlid_results["predicted_top_2"] = openlid_results["top_pred"].apply(
    lambda s: list(ast.literal_eval(s).keys())[:2]
)

In [1061]:
openlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,"{'zho_Hant': 0.5538930892944336, 'zho_Hans': 0...",zho_Hant,0.553893,"[eng_Latn, tam_Taml]","[zho_Hant, zho_Hans]"
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.9959315061569214, 'hrv_Latn': 0...",eus_Latn,0.995932,"[eus_Latn, spa_Latn]","[eus_Latn, hrv_Latn]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.9779373407363892, 'swh_Latn': 0...",eng_Latn,0.977937,"[eng_Latn, hin_Deva]","[eng_Latn, swh_Latn]"
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'yor_Latn': 0.19660769402980804, 'hau_Latn': ...",yor_Latn,0.196608,"[eng_Latn, tam_Taml]","[yor_Latn, hau_Latn]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'zho_Hans': 0.9475940465927124, 'zho_Hant': 0...",zho_Hans,0.947594,"[zho_Hant, eng_Latn]","[zho_Hans, zho_Hant]"
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.8605847358703613, 'dan_Latn': 0...",eng_Latn,0.860585,"[eng_Latn, hin_Deva]","[eng_Latn, dan_Latn]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.30808666348457336, 'est_Latn': ...",eng_Latn,0.308087,"[eng_Latn, tam_Taml]","[eng_Latn, est_Latn]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'zho_Hans': 0.9955909848213196, 'zho_Hant': 0...",zho_Hans,0.995591,"[zho_Hant, eng_Latn]","[zho_Hans, zho_Hant]"
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 1.0000077486038208, 'ajp_Arab': 1...",arz_Arab,1.000008,"[arz_Arab, eng_Latn]","[arz_Arab, ajp_Arab]"


In [1062]:
language_to_iso_script

{'Arabizi': 'arb_Latn',
 'Basque': 'eus_Latn',
 'Chinese': 'zho_Hant',
 'Egyptian': 'arz_Arab',
 'English': 'eng_Latn',
 'German': 'deu_Latn',
 'Hindi': 'hin_Deva',
 'Indonesian': 'ind_Latn',
 'MSA': 'arb_Arab',
 'Malayalam': 'mal_Mlym',
 'Saudi': 'ars_Arab',
 'Spanish': 'spa_Latn',
 'Tamil': 'tam_Taml',
 'Turkish': 'tur_Latn'}

In [1063]:
openlid_to_fasttext_mapping = {}
for key, value in language_to_iso_script.items():
    openlid_to_fasttext_mapping[value] = key

In [1064]:
openlid_to_fasttext_mapping

{'arb_Latn': 'Arabizi',
 'eus_Latn': 'Basque',
 'zho_Hant': 'Chinese',
 'arz_Arab': 'Egyptian',
 'eng_Latn': 'English',
 'deu_Latn': 'German',
 'hin_Deva': 'Hindi',
 'ind_Latn': 'Indonesian',
 'arb_Arab': 'MSA',
 'mal_Mlym': 'Malayalam',
 'ars_Arab': 'Saudi',
 'spa_Latn': 'Spanish',
 'tam_Taml': 'Tamil',
 'tur_Latn': 'Turkish'}

In [1065]:
openlid_results = openlid_results['predicted_top_2']

In [1066]:
openlid_results

0        [zho_Hant, zho_Hans]
1        [eus_Latn, hrv_Latn]
2        [eng_Latn, swh_Latn]
3        [yor_Latn, hau_Latn]
4        [zho_Hans, zho_Hant]
                 ...         
35609    [eng_Latn, dan_Latn]
35610    [eng_Latn, est_Latn]
35611    [zho_Hans, zho_Hant]
35612    [arz_Arab, ajp_Arab]
35613    [eng_Latn, pol_Latn]
Name: predicted_top_2, Length: 35614, dtype: object

In [1067]:
openlid_results = openlid_results.apply(
    lambda x: [openlid_to_fasttext_mapping[label] for label in x if label in openlid_to_fasttext_mapping]
)

In [1068]:
openlid_results

0         [Chinese]
1          [Basque]
2         [English]
3                []
4         [Chinese]
            ...    
35609     [English]
35610     [English]
35611     [Chinese]
35612    [Egyptian]
35613     [English]
Name: predicted_top_2, Length: 35614, dtype: object

In [1069]:
openlid_combined_labels = pd.DataFrame({'true_labels_list': true_labels, 'predicted_labels_list': openlid_results})

In [1070]:
openlid_combined_labels

,true_labels_list,predicted_labels_list
0,"['English', 'Tamil']",[Chinese]
1,"['Basque', 'Spanish']",[Basque]
2,"['English', 'Hindi']",[English]
3,"['English', 'Tamil']",[]
4,"['Chinese', 'English']",[Chinese]
...,...,...
35609,"['English', 'Hindi']",[English]
35610,"['English', 'Tamil']",[English]
35611,"['Chinese', 'English']",[Chinese]
35612,"['Egyptian', 'English']",[Egyptian]


In [1071]:
openlid_combined_labels['true_label'] = openlid_combined_labels['true_labels_list'].apply(lambda x: '-'.join(sorted(ast.literal_eval(x))))
openlid_combined_labels['predicted_label'] = openlid_combined_labels['predicted_labels_list'].apply(lambda x: '-'.join(sorted(x)))

In [1072]:
openlid_combined_labels

,true_labels_list,predicted_labels_list,true_label,predicted_label
0,"['English', 'Tamil']",[Chinese],English-Tamil,Chinese
1,"['Basque', 'Spanish']",[Basque],Basque-Spanish,Basque
2,"['English', 'Hindi']",[English],English-Hindi,English
3,"['English', 'Tamil']",[],English-Tamil,
4,"['Chinese', 'English']",[Chinese],Chinese-English,Chinese
...,...,...,...,...
35609,"['English', 'Hindi']",[English],English-Hindi,English
35610,"['English', 'Tamil']",[English],English-Tamil,English
35611,"['Chinese', 'English']",[Chinese],Chinese-English,Chinese
35612,"['Egyptian', 'English']",[Egyptian],Egyptian-English,Egyptian


In [1073]:
openlid_fpr_per_label = compute_fpr(openlid_combined_labels)
for label, fpr in openlid_fpr_per_label.items():
    print(f"{label}: {fpr:.4f}")

Arabizi-English: 0.0000
Basque-Spanish: 0.0013
Chinese-English: 0.0019
Egyptian-English: 0.0001
Egyptian-MSA: 0.0026
English-German: 0.0003
English-Hindi: 0.0001
English-Indonesian: 0.0042
English-Malayalam: 0.0000
English-Saudi: 0.0000
English-Spanish: 0.0002
English-Tamil: 0.0000
English-Turkish: 0.0010


In [1074]:
openlid_group_accuracies = compute_full_match_per_group(openlid_combined_labels)
for label, acc in openlid_group_accuracies.items():
    print(f"{label}: {acc:.4f}")


Arabizi-English: 0.0000
Basque-Spanish: 0.1602
Chinese-English: 0.0038
Egyptian-English: 0.0224
Egyptian-MSA: 0.0233
English-German: 0.0435
English-Hindi: 0.0000
English-Indonesian: 0.0927
English-Malayalam: 0.0029
English-Saudi: 0.0164
English-Spanish: 0.0063
English-Tamil: 0.0002
English-Turkish: 0.0424


In [1177]:
openlid_partial = compute_partial_but_not_full_match(openlid_combined_labels)

In [1076]:
openlid_top_langs = most_frequent_language_in_predictions(openlid_combined_labels)
for label, (lang) in openlid_top_langs.items():
    print(f"{label}: Most frequently predicted → {lang}")

Basque-Spanish: Most frequently predicted → Basque
English-Hindi: Most frequently predicted → English
Chinese-English: Most frequently predicted → Chinese
English-Spanish: Most frequently predicted → Spanish
Egyptian-English: Most frequently predicted → Egyptian
Egyptian-MSA: Most frequently predicted → Egyptian
English-Tamil: Most frequently predicted → English
English-German: Most frequently predicted → German
English-Turkish: Most frequently predicted → Turkish
English-Saudi: Most frequently predicted → Saudi
English-Malayalam: Most frequently predicted → English
English-Indonesian: Most frequently predicted → Indonesian
Arabizi-English: Most frequently predicted → English


## Franc

In [1077]:
franc_results = pd.read_csv("../../dataset/code-switch/codeswitch_baselines_results/codeswitch/franc_multi_cs.csv")

In [1078]:
language_to_iso_script = {
    'Arabizi': 'arb',
    'Basque': 'eus',
    'Chinese': 'zho', 
    'Egyptian': 'arz',
    'English': 'eng',
    'German': 'deu',
    'Hindi': 'hin',
    'Indonesian': 'ind',
    'MSA': 'arb',       
    'Malayalam': 'mal',
    'Saudi': 'ars',
    'Spanish': 'spa',
    'Tamil': 'tam',
    'Turkish': 'tur'
}

In [1079]:
def map_to_iso_pair(lang_pair):
    langs = lang_pair.split('-')
    return [language_to_iso_script.get(lang, 'unknown') for lang in langs]

In [1080]:
franc_results['iso_pair'] = franc_results['languages'].apply(map_to_iso_pair)

In [1081]:
def get_top2_iso(pred_lang_str):
    try:
        pred_lang_dict = ast.literal_eval(pred_lang_str)
        top2 = sorted(pred_lang_dict.items(), key=lambda x: x[1], reverse=True)[:2]
        top2_langs = [lang for lang, _ in top2]
        return top2_langs
    except Exception as e:
        print(f"Error parsing: {pred_lang_str} — {e}")
        return []

In [1082]:
import ast

franc_results["predicted_top_2"] = franc_results["top_pred"].apply(
    lambda s: list(ast.literal_eval(s).keys())[:2]
)

In [1083]:
franc_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,"{'fao': 1.0, 'hun': 0.9845837615621789, 'mos':...",fao,1.0,"[eng, tam]","[fao, hun]"
1,zein da kanala!,Basque-Spanish,"{'emk': 1.0, 'bam': 0.9340620592383639, 'war':...",emk,1.0,"[eus, spa]","[emk, bam]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng': 1.0, 'sco': 0.9708080780625696, 'pcm':...",eng,1.0,"[eng, hin]","[eng, sco]"
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'buc': 1.0, 'kal': 0.9903674634320371, 'mxv':...",buc,1.0,"[eng, tam]","[buc, kal]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,{'cmn': 1.0},cmn,1.0,"[zho, eng]",[cmn]
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'sco': 1.0, 'rmn': 0.9833437110834371, 'eng':...",sco,1.0,"[eng, hin]","[sco, rmn]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'afr': 1.0, 'kng': 0.9945155393053017, 'cha':...",afr,1.0,"[eng, tam]","[afr, kng]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,{'cmn': 1.0},cmn,1.0,"[zho, eng]",[cmn]
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arb': 1.0, 'zlm': 0.8602086140639816, 'pes':...",arb,1.0,"[arz, eng]","[arb, zlm]"


In [1084]:
franc_to_fasttext_mapping = {}
for key, value in language_to_iso_script.items():
    franc_to_fasttext_mapping[value] = key

In [1085]:
franc_results = franc_results['predicted_top_2']

In [1086]:
franc_results

0        [fao, hun]
1        [emk, bam]
2        [eng, sco]
3        [buc, kal]
4             [cmn]
            ...    
35609    [sco, rmn]
35610    [afr, kng]
35611         [cmn]
35612    [arb, zlm]
35613    [nob, eng]
Name: predicted_top_2, Length: 35614, dtype: object

In [1087]:
franc_results = franc_results.apply(
    lambda x: [franc_to_fasttext_mapping[label] for label in x if label in franc_to_fasttext_mapping]
)

In [1088]:
franc_results

0               []
1               []
2        [English]
3               []
4               []
           ...    
35609           []
35610           []
35611           []
35612        [MSA]
35613    [English]
Name: predicted_top_2, Length: 35614, dtype: object

In [1089]:
franc_combined_labels = pd.DataFrame({'true_labels_list': true_labels, 'predicted_labels_list': franc_results})

In [1090]:
franc_combined_labels

,true_labels_list,predicted_labels_list
0,"['English', 'Tamil']",[]
1,"['Basque', 'Spanish']",[]
2,"['English', 'Hindi']",[English]
3,"['English', 'Tamil']",[]
4,"['Chinese', 'English']",[]
...,...,...
35609,"['English', 'Hindi']",[]
35610,"['English', 'Tamil']",[]
35611,"['Chinese', 'English']",[]
35612,"['Egyptian', 'English']",[MSA]


In [1091]:
for res in franc_results:
    if len(res) > 1:
        print(res)

['German', 'English']
['Spanish', 'English']
['Indonesian', 'Basque']
['German', 'English']
['German', 'English']
['Indonesian', 'Basque']
['English', 'German']
['Basque', 'Indonesian']
['Basque', 'Spanish']
['English', 'Spanish']
['Basque', 'Spanish']
['German', 'English']
['English', 'German']
['English', 'Spanish']
['English', 'Spanish']
['Spanish', 'Basque']
['Spanish', 'English']
['Basque', 'Indonesian']
['Spanish', 'English']
['English', 'Indonesian']
['Indonesian', 'English']
['English', 'Spanish']
['Basque', 'Spanish']
['Basque', 'Indonesian']
['English', 'German']
['Spanish', 'English']
['English', 'German']
['English', 'Spanish']
['English', 'German']
['Basque', 'Spanish']
['English', 'Spanish']
['Spanish', 'Basque']
['English', 'Spanish']
['English', 'German']
['Spanish', 'English']


In [1092]:
franc_combined_labels['true_label'] = franc_combined_labels['true_labels_list'].apply(lambda x: '-'.join(sorted(ast.literal_eval(x))))
franc_combined_labels['predicted_label'] = franc_combined_labels['predicted_labels_list'].apply(lambda x: '-'.join(sorted(x)))

In [1093]:
franc_fpr_per_label = compute_fpr(franc_combined_labels)
for label, fpr in franc_fpr_per_label.items():
    print(f"{label}: {fpr:.4f}")

Arabizi-English: 0.0000
Basque-Spanish: 0.0001
Chinese-English: 0.0000
Egyptian-English: 0.0000
Egyptian-MSA: 0.0000
English-German: 0.0002
English-Hindi: 0.0000
English-Indonesian: 0.0000
English-Malayalam: 0.0000
English-Saudi: 0.0000
English-Spanish: 0.0001
English-Tamil: 0.0000
English-Turkish: 0.0000


In [1094]:
franc_group_accuracies = compute_full_match_per_group(franc_combined_labels)
for label, acc in franc_group_accuracies.items():
    print(f"{label}: {acc:.4f}")


Arabizi-English: 0.0000
Basque-Spanish: 0.0073
Chinese-English: 0.0000
Egyptian-English: 0.0000
Egyptian-MSA: 0.0000
English-German: 0.0119
English-Hindi: 0.0000
English-Indonesian: 0.0081
English-Malayalam: 0.0000
English-Saudi: 0.0000
English-Spanish: 0.0013
English-Tamil: 0.0000
English-Turkish: 0.0000


In [1178]:
franc_partial = compute_partial_but_not_full_match(franc_combined_labels)

In [1096]:
franc_top_langs = most_frequent_language_in_predictions(franc_combined_labels)
for label, (lang) in franc_top_langs.items():
    print(f"{label}: Most frequently predicted → {lang}")

English-Hindi: Most frequently predicted → English
English-Spanish: Most frequently predicted → Spanish
Egyptian-MSA: Most frequently predicted → MSA
Basque-Spanish: Most frequently predicted → Basque
English-German: Most frequently predicted → German
English-Turkish: Most frequently predicted → Turkish
English-Saudi: Most frequently predicted → English
Egyptian-English: Most frequently predicted → English
English-Tamil: Most frequently predicted → English
Chinese-English: Most frequently predicted → English
English-Indonesian: Most frequently predicted → Indonesian
English-Malayalam: Most frequently predicted → English
Arabizi-English: Most frequently predicted → English


## Glotlid

In [1097]:
glotlid_results = pd.read_csv("../../dataset/code-switch/codeswitch_baselines_results/codeswitch/glotlid_multi_cs.csv")

In [1098]:
language_to_iso_script = {
    'Arabizi': 'arbizi_Latn',
    'Basque': 'eus_Latn',
    'Chinese': 'zho_Hans', 
    'Egyptian': 'arz_Arab',
    'English': 'eng_Latn',
    'German': 'deu_Latn',
    'Hindi': 'hin_Deva',
    'Indonesian': 'ind_Latn',
    'MSA': 'arb_Arab',       
    'Malayalam': 'mal_Mlym',
    'Saudi': 'ars_Arab',
    'Spanish': 'spa_Latn',
    'Tamil': 'tam_Taml',
    'Turkish': 'tur_Latn'
}

In [1099]:
def map_to_iso_pair(lang_pair):
    langs = lang_pair.split('-')
    return [language_to_iso_script.get(lang, 'unknown') for lang in langs]

In [1100]:
glotlid_results['iso_pair'] = glotlid_results['languages'].apply(map_to_iso_pair)

In [1101]:
def get_top2_iso(pred_lang_str):
    try:
        pred_lang_dict = ast.literal_eval(pred_lang_str)
        top2 = sorted(pred_lang_dict.items(), key=lambda x: x[1], reverse=True)[:2]
        top2_langs = [lang for lang, _ in top2]
        return top2_langs
    except Exception as e:
        print(f"Error parsing: {pred_lang_str} — {e}")
        return []

In [1102]:
import ast

glotlid_results["predicted_top_2"] = glotlid_results["top_pred"].apply(
    lambda s: list(ast.literal_eval(s).keys())[:2]
)

In [1103]:
glotlid_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,"{'tur_Latn': 0.999435544013977, 'kia_Latn': 0....",tur_Latn,0.999436,"[eng_Latn, tam_Taml]","[tur_Latn, kia_Latn]"
1,zein da kanala!,Basque-Spanish,"{'eus_Latn': 0.5763978362083435, 'bre_Latn': 0...",eus_Latn,0.576398,"[eus_Latn, spa_Latn]","[eus_Latn, bre_Latn]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,"{'eng_Latn': 0.830464780330658, 'pcm_Latn': 0....",eng_Latn,0.830465,"[eng_Latn, hin_Deva]","[eng_Latn, pcm_Latn]"
3,10 million views inniku 10 Mani ku...,English-Tamil,"{'rnd_Latn': 0.30418160557746887, 'pol_Latn': ...",rnd_Latn,0.304182,"[eng_Latn, tam_Taml]","[rnd_Latn, pol_Latn]"
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,"{'cmn_Hani': 0.9767146110534668, 'nan_Hani': 0...",cmn_Hani,0.976715,"[zho_Hans, eng_Latn]","[cmn_Hani, nan_Hani]"
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,"{'eng_Latn': 0.9581118822097778, 'bzj_Latn': 0...",eng_Latn,0.958112,"[eng_Latn, hin_Deva]","[eng_Latn, bzj_Latn]"
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"{'eng_Latn': 0.4568457007408142, 'tel_Latn': 0...",eng_Latn,0.456846,"[eng_Latn, tam_Taml]","[eng_Latn, tel_Latn]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,"{'cmn_Hani': 0.9976520538330078, 'wuu_Hani': 0...",cmn_Hani,0.997652,"[zho_Hans, eng_Latn]","[cmn_Hani, wuu_Hani]"
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,"{'arz_Arab': 0.9949780106544495, 'ajp_Arab': 0...",arz_Arab,0.994978,"[arz_Arab, eng_Latn]","[arz_Arab, ajp_Arab]"


In [1104]:
glotlid_to_fasttext_mapping = {}
for key, value in language_to_iso_script.items():
    glotlid_to_fasttext_mapping[value] = key

In [1105]:
glotlid_results = glotlid_results['predicted_top_2']

In [1106]:
glotlid_results

0        [tur_Latn, kia_Latn]
1        [eus_Latn, bre_Latn]
2        [eng_Latn, pcm_Latn]
3        [rnd_Latn, pol_Latn]
4        [cmn_Hani, nan_Hani]
                 ...         
35609    [eng_Latn, bzj_Latn]
35610    [eng_Latn, tel_Latn]
35611    [cmn_Hani, wuu_Hani]
35612    [arz_Arab, ajp_Arab]
35613    [eng_Latn, kin_Latn]
Name: predicted_top_2, Length: 35614, dtype: object

In [1107]:
glotlid_results = glotlid_results.apply(
    lambda x: [glotlid_to_fasttext_mapping[label] for label in x if label in glotlid_to_fasttext_mapping]
)

In [1108]:
glotlid_results

0         [Turkish]
1          [Basque]
2         [English]
3                []
4                []
            ...    
35609     [English]
35610     [English]
35611            []
35612    [Egyptian]
35613     [English]
Name: predicted_top_2, Length: 35614, dtype: object

In [1109]:
glotlid_combined_labels = pd.DataFrame({'true_labels_list': true_labels, 'predicted_labels_list': glotlid_results})

In [1110]:
glotlid_combined_labels

,true_labels_list,predicted_labels_list
0,"['English', 'Tamil']",[Turkish]
1,"['Basque', 'Spanish']",[Basque]
2,"['English', 'Hindi']",[English]
3,"['English', 'Tamil']",[]
4,"['Chinese', 'English']",[]
...,...,...
35609,"['English', 'Hindi']",[English]
35610,"['English', 'Tamil']",[English]
35611,"['Chinese', 'English']",[]
35612,"['Egyptian', 'English']",[Egyptian]


In [1111]:
glotlid_combined_labels[glotlid_combined_labels['true_labels_list'].apply(lambda x: set(x) == set(['Arabic', 'English']))]

,true_labels_list,predicted_labels_list


In [1112]:
glotlid_combined_labels

,true_labels_list,predicted_labels_list
0,"['English', 'Tamil']",[Turkish]
1,"['Basque', 'Spanish']",[Basque]
2,"['English', 'Hindi']",[English]
3,"['English', 'Tamil']",[]
4,"['Chinese', 'English']",[]
...,...,...
35609,"['English', 'Hindi']",[English]
35610,"['English', 'Tamil']",[English]
35611,"['Chinese', 'English']",[]
35612,"['Egyptian', 'English']",[Egyptian]


In [1113]:
for res in glotlid_results:
    if len(res) > 1:
        print(res)

['MSA', 'Egyptian']
['Egyptian', 'MSA']
['Spanish', 'Turkish']
['German', 'English']
['Egyptian', 'Saudi']
['Basque', 'English']
['Egyptian', 'MSA']
['Egyptian', 'Saudi']
['MSA', 'Egyptian']
['Saudi', 'MSA']
['Turkish', 'Spanish']
['Saudi', 'Egyptian']
['Egyptian', 'MSA']
['Egyptian', 'MSA']
['Spanish', 'English']
['English', 'Spanish']
['Egyptian', 'MSA']
['MSA', 'Egyptian']
['MSA', 'Egyptian']
['English', 'German']
['Egyptian', 'English']
['MSA', 'Saudi']
['Spanish', 'English']
['MSA', 'Egyptian']
['Egyptian', 'MSA']
['English', 'Indonesian']
['Egyptian', 'Saudi']
['Spanish', 'English']
['Egyptian', 'MSA']
['Spanish', 'English']
['Saudi', 'Egyptian']
['Spanish', 'English']
['Saudi', 'Egyptian']
['Spanish', 'English']
['English', 'Turkish']
['Egyptian', 'Saudi']
['Egyptian', 'MSA']
['English', 'Indonesian']
['German', 'English']
['Saudi', 'Egyptian']
['Egyptian', 'Saudi']
['Egyptian', 'Saudi']
['Egyptian', 'Saudi']
['Saudi', 'MSA']
['Saudi', 'Egyptian']
['Egyptian', 'Saudi']
['Spanish

In [1114]:
glotlid_combined_labels['true_label'] = glotlid_combined_labels['true_labels_list'].apply(lambda x: '-'.join(sorted(ast.literal_eval(x))))
glotlid_combined_labels['predicted_label'] = glotlid_combined_labels['predicted_labels_list'].apply(lambda x: '-'.join(sorted(x)))

In [1115]:
glotlid_combined_labels

,true_labels_list,predicted_labels_list,true_label,predicted_label
0,"['English', 'Tamil']",[Turkish],English-Tamil,Turkish
1,"['Basque', 'Spanish']",[Basque],Basque-Spanish,Basque
2,"['English', 'Hindi']",[English],English-Hindi,English
3,"['English', 'Tamil']",[],English-Tamil,
4,"['Chinese', 'English']",[],Chinese-English,
...,...,...,...,...
35609,"['English', 'Hindi']",[English],English-Hindi,English
35610,"['English', 'Tamil']",[English],English-Tamil,English
35611,"['Chinese', 'English']",[],Chinese-English,
35612,"['Egyptian', 'English']",[Egyptian],Egyptian-English,Egyptian


In [1116]:
for res in glotlid_combined_labels['predicted_label']:
    if res=='Arabizi-English':
        print(res)

In [1117]:
glotlid_fpr_per_label = compute_fpr(glotlid_combined_labels)
for label, fpr in glotlid_fpr_per_label.items():
    print(f"{label}: {fpr:.4f}")

Arabizi-English: 0.0000
Basque-Spanish: 0.0003
Chinese-English: 0.0000
Egyptian-English: 0.0000
Egyptian-MSA: 0.0043
English-German: 0.0021
English-Hindi: 0.0000
English-Indonesian: 0.0027
English-Malayalam: 0.0000
English-Saudi: 0.0000
English-Spanish: 0.0003
English-Tamil: 0.0000
English-Turkish: 0.0014


In [1118]:
glotlid_group_accuracies = compute_full_match_per_group(glotlid_combined_labels)
for label, acc in glotlid_group_accuracies.items():
    print(f"{label}: {acc:.4f}")


Arabizi-English: 0.0000
Basque-Spanish: 0.1189
Chinese-English: 0.0000
Egyptian-English: 0.0109
Egyptian-MSA: 0.0493
English-German: 0.0632
English-Hindi: 0.0000
English-Indonesian: 0.1008
English-Malayalam: 0.0029
English-Saudi: 0.0047
English-Spanish: 0.0374
English-Tamil: 0.0005
English-Turkish: 0.0424


In [1179]:
glotlid_partial = compute_partial_but_not_full_match(glotlid_combined_labels)

In [1120]:
glotlid_top_langs = most_frequent_language_in_predictions(glotlid_combined_labels)
for label, (lang) in glotlid_top_langs.items():
    print(f"{label}: Most frequently predicted → {lang}")

Basque-Spanish: Most frequently predicted → Basque
English-Hindi: Most frequently predicted → English
English-Spanish: Most frequently predicted → Spanish
Egyptian-English: Most frequently predicted → Egyptian
English-Tamil: Most frequently predicted → English
Egyptian-MSA: Most frequently predicted → Egyptian
English-German: Most frequently predicted → German
English-Turkish: Most frequently predicted → Turkish
English-Saudi: Most frequently predicted → Saudi
English-Malayalam: Most frequently predicted → English
Arabizi-English: Most frequently predicted → English
English-Indonesian: Most frequently predicted → Indonesian
Chinese-English: Most frequently predicted → English


## Langdetect

In [1121]:
langdetect_results = pd.read_csv("../../dataset/code-switch/codeswitch_baselines_results/codeswitch/langdetect_multi_cs.csv")

In [1122]:
langdetect_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob
0,Yellam Avan seyal . . .,English-Tamil,[tr:0.9999969491798868],tr,0.999997
1,zein da kanala!,Basque-Spanish,"[id:0.5714266758490081, tr:0.2857148622637077,...",id,0.571427
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,[en:0.9999962959360083],en,0.999996
3,10 million views inniku 10 Mani ku...,English-Tamil,[et:0.999997184566695],et,0.999997
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,[en:0.9999964996843613],en,0.999996
...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,[en:0.9999975749447501],en,0.999998
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"[no:0.7142823351653185, et:0.28571567140402626]",no,0.714282
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,[zh-cn:0.9999982567826919],zh-cn,0.999998
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,[ar:0.9999992294683185],ar,0.999999


In [1123]:
language_to_iso_script = {
    'Arabizi': 'arb',
    'Basque': 'eu',
    'Chinese': 'zh', 
    'Egyptian': 'eg',
    'English': 'en',
    'German': 'de',
    'Hindi': 'hi',
    'Indonesian': 'in',
    'MSA': 'ar',       
    'Malayalam': 'ml',
    'Saudi': 'ars',
    'Spanish': 'es',
    'Tamil': 'ta',
    'Turkish': 'tr'
}

In [1124]:
def map_to_iso_pair(lang_pair):
    langs = lang_pair.split('-')
    return [language_to_iso_script.get(lang, 'unknown') for lang in langs]

In [1125]:
langdetect_results['iso_pair'] = langdetect_results['languages'].apply(map_to_iso_pair)

In [1126]:
def get_top2_iso(pred_lang_str):
    try:
        pred_lang_dict = ast.literal_eval(pred_lang_str)
        top2 = sorted(pred_lang_dict.items(), key=lambda x: x[1], reverse=True)[:2]
        top2_langs = [lang for lang, _ in top2]
        return top2_langs
    except Exception as e:
        print(f"Error parsing: {pred_lang_str} — {e}")
        return []

In [1127]:
import re

langdetect_results["predicted_top_2"] = langdetect_results["top_pred"].apply(
    lambda s: re.findall(r'(\w+):', s)[:2] if isinstance(s, str) else []
)


In [1128]:
langdetect_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,[tr:0.9999969491798868],tr,0.999997,"[en, ta]",[tr]
1,zein da kanala!,Basque-Spanish,"[id:0.5714266758490081, tr:0.2857148622637077,...",id,0.571427,"[eu, es]","[id, tr]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,[en:0.9999962959360083],en,0.999996,"[en, hi]",[en]
3,10 million views inniku 10 Mani ku...,English-Tamil,[et:0.999997184566695],et,0.999997,"[en, ta]",[et]
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,[en:0.9999964996843613],en,0.999996,"[zh, en]",[en]
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,[en:0.9999975749447501],en,0.999998,"[en, hi]",[en]
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"[no:0.7142823351653185, et:0.28571567140402626]",no,0.714282,"[en, ta]","[no, et]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,[zh-cn:0.9999982567826919],zh-cn,0.999998,"[zh, en]",[cn]
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,[ar:0.9999992294683185],ar,0.999999,"[eg, en]",[ar]


In [1129]:
langdetect_results

,codeswitch_sentence,languages,top_pred,pred_lang,pred_prob,iso_pair,predicted_top_2
0,Yellam Avan seyal . . .,English-Tamil,[tr:0.9999969491798868],tr,0.999997,"[en, ta]",[tr]
1,zein da kanala!,Basque-Spanish,"[id:0.5714266758490081, tr:0.2857148622637077,...",id,0.571427,"[eu, es]","[id, tr]"
2,"Jise unhone mera greatest fat kaha tha, usne u...",English-Hindi,[en:0.9999962959360083],en,0.999996,"[en, hi]",[en]
3,10 million views inniku 10 Mani ku...,English-Tamil,[et:0.999997184566695],et,0.999997,"[en, ta]",[et]
4,在 2020 年 11 月的投票中，加州选民通过了 Proposition 22。该法案让零...,Chinese-English,[en:0.9999964996843613],en,0.999996,"[zh, en]",[en]
...,...,...,...,...,...,...,...
35609,har chiz harkat mai thi- up and down and sidew...,English-Hindi,[en:0.9999975749447501],en,0.999998,"[en, hi]",[en]
35610,Hello Telugu fans yavaraina unte like veskond...,English-Tamil,"[no:0.7142823351653185, et:0.28571567140402626]",no,0.714282,"[en, ta]","[no, et]"
35611,日本软银投资的 AI 创业公司 Engineer.ai 提供人工辅助的 AI 去实现移动 A...,Chinese-English,[zh-cn:0.9999982567826919],zh-cn,0.999998,"[zh, en]",[cn]
35612,يعنى أنا من زمان مش بحب ال .. ال لا أسمع أغانى...,Egyptian-English,[ar:0.9999992294683185],ar,0.999999,"[eg, en]",[ar]


In [1130]:
langdetect_to_fasttext_mapping = {}
for key, value in language_to_iso_script.items():
    langdetect_to_fasttext_mapping[value] = key

In [1131]:
langdetect_results = langdetect_results['predicted_top_2']

In [1132]:
langdetect_results

0            [tr]
1        [id, tr]
2            [en]
3            [et]
4            [en]
           ...   
35609        [en]
35610    [no, et]
35611        [cn]
35612        [ar]
35613        [en]
Name: predicted_top_2, Length: 35614, dtype: object

In [1133]:
langdetect_results = langdetect_results.apply(
    lambda x: [langdetect_to_fasttext_mapping[label] for label in x if label in langdetect_to_fasttext_mapping]
)

In [1134]:
langdetect_combined_labels = pd.DataFrame({'true_labels_list': true_labels, 'predicted_labels_list': langdetect_results})

In [1135]:
langdetect_combined_labels

,true_labels_list,predicted_labels_list
0,"['English', 'Tamil']",[Turkish]
1,"['Basque', 'Spanish']",[Turkish]
2,"['English', 'Hindi']",[English]
3,"['English', 'Tamil']",[]
4,"['Chinese', 'English']",[English]
...,...,...
35609,"['English', 'Hindi']",[English]
35610,"['English', 'Tamil']",[]
35611,"['Chinese', 'English']",[]
35612,"['Egyptian', 'English']",[MSA]


In [1136]:
for res in langdetect_results:
    if len(res) > 1:
        print(res)

['English', 'MSA']
['Spanish', 'English']
['Spanish', 'English']
['English', 'MSA']
['English', 'MSA']
['MSA', 'English']
['MSA', 'English']
['MSA', 'English']
['English', 'MSA']
['English', 'Spanish']
['English', 'Spanish']
['English', 'Spanish']
['Spanish', 'Turkish']
['Spanish', 'English']
['Spanish', 'English']
['German', 'Spanish']
['MSA', 'English']
['Spanish', 'English']
['English', 'Spanish']
['MSA', 'English']
['Spanish', 'English']
['Spanish', 'English']
['English', 'German']
['English', 'Spanish']
['Spanish', 'English']
['English', 'Spanish']
['MSA', 'English']
['MSA', 'English']
['MSA', 'English']
['MSA', 'English']
['Spanish', 'English']
['English', 'Spanish']
['Spanish', 'German']
['English', 'Spanish']
['Spanish', 'English']
['German', 'English']
['German', 'English']
['Spanish', 'English']
['English', 'Spanish']
['Spanish', 'English']
['Spanish', 'English']
['MSA', 'English']
['Spanish', 'English']
['MSA', 'English']
['MSA', 'English']
['English', 'MSA']
['German', 'Eng

In [1137]:
langdetect_combined_labels['true_label'] = langdetect_combined_labels['true_labels_list'].apply(lambda x: '-'.join(sorted(ast.literal_eval(x))))
langdetect_combined_labels['predicted_label'] = langdetect_combined_labels['predicted_labels_list'].apply(lambda x: '-'.join(sorted(x)))

In [1138]:
langdetect_fpr_per_label = compute_fpr(langdetect_combined_labels)
for label, fpr in langdetect_fpr_per_label.items():
    print(f"{label}: {fpr:.4f}")

Arabizi-English: 0.0000
Basque-Spanish: 0.0000
Chinese-English: 0.0000
Egyptian-English: 0.0000
Egyptian-MSA: 0.0000
English-German: 0.0019
English-Hindi: 0.0000
English-Indonesian: 0.0000
English-Malayalam: 0.0000
English-Saudi: 0.0000
English-Spanish: 0.0006
English-Tamil: 0.0000
English-Turkish: 0.0011


In [1139]:
langdetect_group_accuracies = compute_full_match_per_group(langdetect_combined_labels)
for label, acc in langdetect_group_accuracies.items():
    print(f"{label}: {acc:.4f}")


Arabizi-English: 0.0000
Basque-Spanish: 0.0000
Chinese-English: 0.0000
Egyptian-English: 0.0000
Egyptian-MSA: 0.0000
English-German: 0.0514
English-Hindi: 0.0001
English-Indonesian: 0.0000
English-Malayalam: 0.0006
English-Saudi: 0.0000
English-Spanish: 0.0647
English-Tamil: 0.0005
English-Turkish: 0.0254


In [1180]:
langdetect_partial = compute_partial_but_not_full_match(langdetect_combined_labels)

In [1141]:
langdetect_top_langs = most_frequent_language_in_predictions(langdetect_combined_labels)
for label, (lang) in langdetect_top_langs.items():
    print(f"{label}: Most frequently predicted → {lang}")

English-Hindi: Most frequently predicted → English
Chinese-English: Most frequently predicted → English
English-Spanish: Most frequently predicted → Spanish
Egyptian-MSA: Most frequently predicted → MSA
English-Malayalam: Most frequently predicted → English
English-Tamil: Most frequently predicted → English
Egyptian-English: Most frequently predicted → English
English-German: Most frequently predicted → German
English-Turkish: Most frequently predicted → Turkish
English-Saudi: Most frequently predicted → English
Arabizi-English: Most frequently predicted → English
English-Indonesian: Most frequently predicted → English
Basque-Spanish: Most frequently predicted → Spanish


## Code-Switching Model

In [1142]:
cs_model_preds = pd.read_csv('fasttext_multilabel_predictions.csv')['predicted_languages']

In [1143]:
cs_model_combined_labels = pd.DataFrame({'true_labels_list': true_labels, 'predicted_labels_list': cs_model_preds})

In [1144]:
cs_model_combined_labels

,true_labels_list,predicted_labels_list
0,"['English', 'Tamil']","['English', 'Tamil']"
1,"['Basque', 'Spanish']",[]
2,"['English', 'Hindi']","['English', 'Hindi']"
3,"['English', 'Tamil']","['English', 'Tamil']"
4,"['Chinese', 'English']","['Chinese', 'English']"
...,...,...
35609,"['English', 'Hindi']","['English', 'Hindi']"
35610,"['English', 'Tamil']","['English', 'Tamil']"
35611,"['Chinese', 'English']","['Chinese', 'English']"
35612,"['Egyptian', 'English']","['Egyptian', 'English']"


In [1145]:
cs_model_combined_labels['true_label'] = cs_model_combined_labels['true_labels_list'].apply(lambda x: '-'.join(sorted(ast.literal_eval(x))))
cs_model_combined_labels['predicted_label'] = cs_model_combined_labels['predicted_labels_list'].apply(lambda x: '-'.join(sorted(ast.literal_eval(x))))

In [1146]:
cs_model_combined_labels

,true_labels_list,predicted_labels_list,true_label,predicted_label
0,"['English', 'Tamil']","['English', 'Tamil']",English-Tamil,English-Tamil
1,"['Basque', 'Spanish']",[],Basque-Spanish,
2,"['English', 'Hindi']","['English', 'Hindi']",English-Hindi,English-Hindi
3,"['English', 'Tamil']","['English', 'Tamil']",English-Tamil,English-Tamil
4,"['Chinese', 'English']","['Chinese', 'English']",Chinese-English,Chinese-English
...,...,...,...,...
35609,"['English', 'Hindi']","['English', 'Hindi']",English-Hindi,English-Hindi
35610,"['English', 'Tamil']","['English', 'Tamil']",English-Tamil,English-Tamil
35611,"['Chinese', 'English']","['Chinese', 'English']",Chinese-English,Chinese-English
35612,"['Egyptian', 'English']","['Egyptian', 'English']",Egyptian-English,Egyptian-English


In [1147]:
cs_fpr_per_label = compute_fpr(cs_model_combined_labels)
for label, fpr in cs_fpr_per_label.items():
    print(f"{label}: {fpr:.4f}")

Arabizi-English: 0.0000
Basque-Spanish: 0.0000
Chinese-English: 0.0001
Egyptian-English: 0.0011
Egyptian-MSA: 0.0057
English-German: 0.0004
English-Hindi: 0.0017
English-Indonesian: 0.0000
English-Malayalam: 0.0004
English-Saudi: 0.0001
English-Spanish: 0.0006
English-Tamil: 0.0010
English-Turkish: 0.0000


In [1148]:
cs_group_accuracies = compute_full_match_per_group(cs_model_combined_labels)
for label, acc in cs_group_accuracies.items():
    print(f"{label}: {acc:.4f}")


Arabizi-English: 0.2555
Basque-Spanish: 0.3835
Chinese-English: 0.8638
Egyptian-English: 0.7238
Egyptian-MSA: 0.7875
English-German: 1.0000
English-Hindi: 0.9504
English-Indonesian: 0.3871
English-Malayalam: 0.6661
English-Saudi: 0.5714
English-Spanish: 0.8123
English-Tamil: 0.8656
English-Turkish: 0.0000


In [1181]:
cs_partial = compute_partial_but_not_full_match(cs_model_combined_labels)

In [1150]:
cs_top_langs = most_frequent_language_in_predictions(cs_model_combined_labels)
for label, (lang) in cs_top_langs.items():
    print(f"{label}: Most frequently predicted → {lang}")

English-Tamil: Most frequently predicted → English
English-Spanish: Most frequently predicted → English
Egyptian-English: Most frequently predicted → Egyptian
Basque-Spanish: Most frequently predicted → Spanish
English-Saudi: Most frequently predicted → English
Chinese-English: Most frequently predicted → English
English-Turkish: Most frequently predicted → English
English-Hindi: Most frequently predicted → English
Arabizi-English: Most frequently predicted → English
English-Malayalam: Most frequently predicted → English
English-Indonesian: Most frequently predicted → English
Egyptian-MSA: Most frequently predicted → Egyptian


## Finetuned OpenLID

In [1151]:
finetuned_model_preds = pd.read_csv('openlid_finetuned_predictions.csv')['predicted_languages']

In [1152]:
finetuned_model_preds

0                              []
1        ['deu_Latn', 'eng_Latn']
2                    ['eng_Latn']
3                    ['eng_Latn']
4                              []
                   ...           
35609                ['eng_Latn']
35610    ['cat_Latn', 'hin_Deva']
35611    ['eng_Latn', 'hin_Deva']
35612                          []
35613                          []
Name: predicted_languages, Length: 35614, dtype: object

In [1153]:
label_mapping = {
    "Arabizi": "arb_Latn", "Basque": "eus_Latn", "Catalan": "cat_Latn",
    "Chinese": "zho_Hans", "Egyptian": "arz_Arab", "English": "eng_Latn",
    "German": "deu_Latn", "Hindi": "hin_Deva", "Indonesian": "ind_Latn",
    "MSA": "arb_Arab", "Malayalam": "mal_Mlym", "Saudi": "ars_Arab",
    "Spanish": "spa_Latn", "Tamil": "tam_Taml", "Turkish": "tur_Latn"
}


In [1154]:
openlid_to_fasttext_mapping = {}
for key, value in label_mapping.items():
    openlid_to_fasttext_mapping[value] = key

In [1155]:
openlid_to_fasttext_mapping['deu_Latn']

'German'

In [1156]:
finetuned_model_preds = finetuned_model_preds.apply(
    lambda x: [openlid_to_fasttext_mapping[label] for label in ast.literal_eval(x)]
)

In [1157]:
finetuned_model_preds

0                       []
1        [German, English]
2                [English]
3                [English]
4                       []
               ...        
35609            [English]
35610     [Catalan, Hindi]
35611     [English, Hindi]
35612                   []
35613                   []
Name: predicted_languages, Length: 35614, dtype: object

In [1158]:
finetuned_model_combined_labels = pd.DataFrame({'true_labels_list': true_labels, 'predicted_labels_list': finetuned_model_preds})

In [1159]:
finetuned_model_combined_labels

,true_labels_list,predicted_labels_list
0,"['English', 'Tamil']",[]
1,"['Basque', 'Spanish']","[German, English]"
2,"['English', 'Hindi']",[English]
3,"['English', 'Tamil']",[English]
4,"['Chinese', 'English']",[]
...,...,...
35609,"['English', 'Hindi']",[English]
35610,"['English', 'Tamil']","[Catalan, Hindi]"
35611,"['Chinese', 'English']","[English, Hindi]"
35612,"['Egyptian', 'English']",[]


In [1160]:
finetuned_model_combined_labels['true_label'] = finetuned_model_combined_labels['true_labels_list'].apply(lambda x: '-'.join(sorted(ast.literal_eval(x))))
finetuned_model_combined_labels['predicted_label'] = finetuned_model_combined_labels['predicted_labels_list'].apply(lambda x: '-'.join(sorted(x)))

In [1161]:
finetuned_model_combined_labels

,true_labels_list,predicted_labels_list,true_label,predicted_label
0,"['English', 'Tamil']",[],English-Tamil,
1,"['Basque', 'Spanish']","[German, English]",Basque-Spanish,English-German
2,"['English', 'Hindi']",[English],English-Hindi,English
3,"['English', 'Tamil']",[English],English-Tamil,English
4,"['Chinese', 'English']",[],Chinese-English,
...,...,...,...,...
35609,"['English', 'Hindi']",[English],English-Hindi,English
35610,"['English', 'Tamil']","[Catalan, Hindi]",English-Tamil,Catalan-Hindi
35611,"['Chinese', 'English']","[English, Hindi]",Chinese-English,English-Hindi
35612,"['Egyptian', 'English']",[],Egyptian-English,


In [1162]:
finetuned_model_combined_labels['predicted_labels_list'] = finetuned_model_combined_labels['predicted_labels_list'].apply(
    lambda x: [str(label) for label in x]
)

In [1163]:
finetuned_model_combined_labels

,true_labels_list,predicted_labels_list,true_label,predicted_label
0,"['English', 'Tamil']",[],English-Tamil,
1,"['Basque', 'Spanish']","[German, English]",Basque-Spanish,English-German
2,"['English', 'Hindi']",[English],English-Hindi,English
3,"['English', 'Tamil']",[English],English-Tamil,English
4,"['Chinese', 'English']",[],Chinese-English,
...,...,...,...,...
35609,"['English', 'Hindi']",[English],English-Hindi,English
35610,"['English', 'Tamil']","[Catalan, Hindi]",English-Tamil,Catalan-Hindi
35611,"['Chinese', 'English']","[English, Hindi]",Chinese-English,English-Hindi
35612,"['Egyptian', 'English']",[],Egyptian-English,


In [1164]:
ft_fpr_per_label = compute_fpr(finetuned_model_combined_labels)
for label, fpr in ft_fpr_per_label.items():
    print(f"{label}: {fpr:.4f}")

Arabizi-English: 0.0000
Basque-Spanish: 0.0000
Chinese-English: 0.0000
Egyptian-English: 0.0000
Egyptian-MSA: 0.0000
English-German: 0.0670
English-Hindi: 0.0442
English-Indonesian: 0.0000
English-Malayalam: 0.0000
English-Saudi: 0.0000
English-Spanish: 0.0000
English-Tamil: 0.0000
English-Turkish: 0.0000


In [1165]:
ft_group_accuracies = compute_full_match_per_group(finetuned_model_combined_labels)
for label, acc in ft_group_accuracies.items():
    print(f"{label}: {acc:.4f}")


Arabizi-English: 0.0000
Basque-Spanish: 0.0000
Chinese-English: 0.0000
Egyptian-English: 0.0000
Egyptian-MSA: 0.0000
English-German: 0.6364
English-Hindi: 0.0341
English-Indonesian: 0.0000
English-Malayalam: 0.0000
English-Saudi: 0.0000
English-Spanish: 0.0000
English-Tamil: 0.0000
English-Turkish: 0.0000


In [1166]:
ft_partial = compute_partial_but_not_full_match(finetuned_model_combined_labels)

In [1167]:
ft_top_langs = most_frequent_language_in_predictions(finetuned_model_combined_labels)
for label, (lang) in ft_top_langs.items():
    print(f"{label}: Most frequently predicted → {lang}")

English-Hindi: Most frequently predicted → English
English-Tamil: Most frequently predicted → English
English-Spanish: Most frequently predicted → English
Chinese-English: Most frequently predicted → English
Egyptian-English: Most frequently predicted → English
English-German: Most frequently predicted → English
English-Turkish: Most frequently predicted → English
English-Saudi: Most frequently predicted → English
English-Malayalam: Most frequently predicted → English
Arabizi-English: Most frequently predicted → English
English-Indonesian: Most frequently predicted → English
Basque-Spanish: Most frequently predicted → Spanish
Egyptian-MSA: Most frequently predicted → MSA


## Tables

In [1169]:
def build_grouped_df(**kwargs):
    all_keys = set().union(*[d.keys() for d in kwargs.values()])
    data = {key: {metric_name: d.get(key, None) for metric_name, d in kwargs.items()} for key in all_keys}
    return pd.DataFrame.from_dict(data, orient='index')

In [1170]:
fpr_metrics = build_grouped_df(
    cs=cs_fpr_per_label,
    ft=ft_fpr_per_label,
    openlid= openlid_fpr_per_label,
    franc= franc_fpr_per_label,
    glotlid= glotlid_fpr_per_label,
    langdetect= langdetect_fpr_per_label,
)

In [1171]:
fpr_metrics

,cs,ft,openlid,franc,glotlid,langdetect
Egyptian-English,0.001132,0.000000,0.000122,0.000000,0.000000,0.000000
Chinese-English,0.000103,0.000034,0.001859,0.000000,0.000000,0.000000
Arabizi-English,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
English-Indonesian,0.000000,0.000000,0.004157,0.000000,0.002658,0.000000
English-Spanish,0.000590,0.000000,0.000174,0.000104,0.000312,0.000590
English-Tamil,0.001021,0.000000,0.000000,0.000000,0.000000,0.000000
English-German,0.000396,0.067023,0.000255,0.000198,0.002121,0.001866
Egyptian-MSA,0.005715,0.000000,0.002604,0.000000,0.004350,0.000000
English-Saudi,0.000142,0.000000,0.000000,0.000000,0.000000,0.000000
Basque-Spanish,0.000000,0.000000,0.001335,0.000085,0.000312,0.000000


In [1172]:
full_match_metrics = build_grouped_df(
    cs=cs_group_accuracies,
    ft=ft_group_accuracies,
    openlid= openlid_group_accuracies,
    franc= franc_group_accuracies,
    glotlid= glotlid_group_accuracies,
    langdetect= langdetect_group_accuracies,
)

In [1173]:
full_match_metrics

,cs,ft,openlid,franc,glotlid,langdetect
Egyptian-English,0.723810,0.000000,0.022449,0.000000,0.010884,0.000000
Chinese-English,0.863782,0.000000,0.003809,0.000000,0.000000,0.000000
Arabizi-English,0.255521,0.000000,0.000000,0.000000,0.000000,0.000000
English-Indonesian,0.387097,0.000000,0.092742,0.008065,0.100806,0.000000
English-Spanish,0.812270,0.000000,0.006326,0.001324,0.037369,0.064734
English-Tamil,0.865574,0.000000,0.000234,0.000000,0.000468,0.000468
English-German,1.000000,0.636364,0.043478,0.011858,0.063241,0.051383
Egyptian-MSA,0.787518,0.000000,0.023312,0.000000,0.049296,0.000000
English-Saudi,0.571429,0.000000,0.016393,0.000000,0.004684,0.000000
Basque-Spanish,0.383495,0.000000,0.160194,0.007282,0.118932,0.000000


In [1183]:
partial_match_metrics = build_grouped_df(
    cs=cs_partial,
    ft=ft_partial,
    openlid= openlid_partial,
    franc= franc_partial,
    glotlid= glotlid_partial,
    langdetect= langdetect_partial,
)

In [1184]:
partial_match_metrics

,cs,ft,openlid,franc,glotlid,langdetect
Egyptian-English,0.464147,0.462751,0.460123,0.500000,0.466563,0.452641
Chinese-English,0.498677,0.439583,0.499786,0.498039,0.497009,0.497413
Arabizi-English,0.480159,0.447791,0.490476,0.500000,0.498282,0.498851
English-Indonesian,0.495305,0.463054,0.496633,0.500000,0.498273,0.494350
English-Tamil,0.493994,0.438043,0.487078,0.500000,0.489362,0.493778
English-Spanish,0.495891,0.455804,0.496265,0.499796,0.497442,0.498934
English-German,NaN,0.478261,0.453665,0.500000,0.500000,0.500000
Egyptian-MSA,0.499441,0.435897,0.486064,0.500000,0.488483,0.500000
Basque-Spanish,0.496124,0.333333,0.494511,0.496246,0.495652,0.497549
English-Saudi,0.495402,0.414462,0.457341,0.500000,0.458068,0.437500


In [1176]:
most_frequent_language_in_predictions = build_grouped_df(
    cs=cs_top_langs,
    ft=ft_top_langs,
    openlid= openlid_top_langs,
    franc= franc_top_langs,
    glotlid= glotlid_top_langs,
    langdetect= langdetect_top_langs,
)
most_frequent_language_in_predictions

,cs,ft,openlid,franc,glotlid,langdetect
Egyptian-English,Egyptian,English,Egyptian,English,Egyptian,English
Chinese-English,English,English,Chinese,English,English,English
Arabizi-English,English,English,English,English,English,English
English-Indonesian,English,English,Indonesian,Indonesian,Indonesian,English
English-Tamil,English,English,English,English,English,English
English-Spanish,English,English,Spanish,Spanish,Spanish,Spanish
English-German,None,English,German,German,German,German
Egyptian-MSA,Egyptian,MSA,Egyptian,MSA,Egyptian,MSA
Basque-Spanish,Spanish,Spanish,Basque,Basque,Basque,Spanish
English-Saudi,English,English,Saudi,English,Saudi,English
